In [3]:
!pip install pytorchvideo torchvision av

In [4]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive

from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
# Connect to Google Drive
drive.mount('/content/drive', force_remount=True)

CLASSES = ['walking', 'running']
NUM_FRAMES = 16  # X3D defaults to 16 frames
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- THE MAGIC THRESHOLD ---
# Anything below 99% confidence gets sorted into Miscellaneous
THRESHOLD = 0.99

# The Paths
# Updated to match the X3D model name we saved during training
MODEL_WEIGHTS_PATH = "/content/drive/MyDrive/video_x3d_TrainedModel.pth"
TEST_FOLDER_PATH = "/content/drive/MyDrive/video_data/test"
RESULT_FOLDER_PATH = "/content/drive/MyDrive/Result"

# Mapping the guesses to your folders (Added the Miscellaneous bucket!)
BUCKET_NAMES = {
    'walking': 'Predicted_Walking',
    'running': 'Predicted_Running',
    'miscellaneous': 'Predicted_Miscellaneous'
}

# --- 3. BUILD THE ROBOT BODY & LOAD THE BRAIN ---
print("Building the robot (X3D-M) and loading the brain...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_m', pretrained=False)

# Change the head so it knows we only have 2 classes
in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, len(CLASSES))

model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# --- 4. VIDEO GLASSES (TRANSFORMS) ---
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0),
            NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]), # X3D specifics
            Resize((256, 256)) # X3D specifics
        ])
    )
])

# --- 5. THE SORTING LOOP ---
print(f"\n--- STARTING THE SORTING MACHINE (Threshold: {THRESHOLD*100}%) ---")

# Look at every single file inside the test folder
for filename in os.listdir(TEST_FOLDER_PATH):

    # Only look at video files, ignore random hidden files
    if not filename.lower().endswith(('.mp4', '.avi', '.mov')):
        continue

    video_path = os.path.join(TEST_FOLDER_PATH, filename)
    print(f"\nWatching: {filename}...")

    try:
        # Load the 2-second clip and put the glasses on it
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)
        video_data = video_transform(video_data)
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        # Make the guess!
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # --- THE MISCELLANEOUS INTERCEPT ---
        confidence_score = conf.item()
        original_guess = CLASSES[pred_idx.item()]

        if confidence_score < THRESHOLD:
            winning_guess = 'miscellaneous'
            print(f"Guess: MISCELLANEOUS (Unsure about {original_guess.upper()}, only {confidence_score*100:.2f}% sure)")
        else:
            winning_guess = original_guess
            print(f"Guess: {winning_guess.upper()} ({confidence_score*100:.2f}% sure)")

        # --- THE FILE MOVING MAGIC ---
        # Look up the exact folder name from your dictionary
        target_bucket = BUCKET_NAMES[winning_guess]

        # Build the paths
        target_folder = os.path.join(RESULT_FOLDER_PATH, target_bucket)
        target_path = os.path.join(target_folder, filename)

        # ⭐️ IMPORTANT: Make sure the folder exists!
        # If 'Predicted_Miscellaneous' isn't in your drive yet, this line creates it automatically.
        os.makedirs(target_folder, exist_ok=True)

        # Copy the video into your folder!
        shutil.copy(video_path, target_path)
        print(f"-> Safely copied into {target_bucket}!")

    except Exception as e:
        print(f"Oh no! Something went wrong with {filename}: {e}")

print("\n--- ALL DONE! CHECK YOUR RESULT FOLDERS! ---")

Mounted at /content/drive
Building the robot (X3D-M) and loading the brain...
Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip

--- STARTING THE SORTING MACHINE (Threshold: 99.0%) ---

Watching: v_PlayingCello_g04_c02.avi...
Guess: MISCELLANEOUS (Unsure about RUNNING, only 71.56% sure)
-> Safely copied into Predicted_Miscellaneous!

Watching: v_TennisSwing_g05_c07.avi...
Guess: MISCELLANEOUS (Unsure about WALKING, only 71.47% sure)
-> Safely copied into Predicted_Miscellaneous!

Watching: v_ShavingBeard_g03_c01.avi...
Guess: MISCELLANEOUS (Unsure about RUNNING, only 64.13% sure)
-> Safely copied into Predicted_Miscellaneous!

--- ALL DONE! CHECK YOUR RESULT FOLDERS! ---
